# Setup

In [1]:
%pip install --upgrade --quiet  langchain langchain-community langchain-ollama langchain-experimental neo4j tiktoken yfiles_jupyter_graphs python-dotenv json-repair langchain-openai langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 

In [2]:
from langchain_core.runnables import  RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser
from langchain_community.graphs import Neo4jGraph
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.chat_models import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores import Neo4jVector
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_ollama import OllamaEmbeddings
import os
from langchain_experimental.llms.ollama_functions import OllamaFunctions
from neo4j import  Driver

from dotenv import load_dotenv

load_dotenv()

False

In [ ]:
import os

os.environ["NEO4J_URI"] = ""
os.environ["NEO4J_USERNAME"] = ""
os.environ["NEO4J_PASSWORD"] = 


graph = Neo4jGraph(url="neo4j+s://b44e89f3.databases.neo4j.io",
                   username="b44e89f3",
                   password="",
                   database="b44e89f3")

/tmp/ipykernel_5634/3789063813.py:14: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url="neo4j+s://b44e89f3.databases.neo4j.io",


In [6]:
!pip install -U langchain-ollama

# Download dataset

In [22]:
import requests
import zipfile
import re

def get_ai_companies(limit=10):
    """Retrieves a list of article titles from the AI companies category."""
    url = "https://en.wikipedia.org/w/api.php"
    headers = {"User-Agent": "MyAICompanyCrawler/1.0 (learning_script)"}
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Artificial intelligence companies",
        "cmlimit": limit,
        "format": "json",
        "cmnamespace": 0
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()

    companies = []
    if "query" in data and "categorymembers" in data["query"]:
        for member in data["query"]["categorymembers"]:
            companies.append(member["title"])

    return companies

def get_article_text(title):
    """Retrieves the full plain text extract of a specific Wikipedia article."""
    url = "https://en.wikipedia.org/w/api.php"
    headers = {"User-Agent": "MyAICompanyCrawler/1.0 (learning_script)"}

    params = {
        "action": "query",
        "prop": "extracts",
        "titles": title,
        "format": "json",
        "explaintext": True,
        "exintro": False     # Set to False to get the FULL article text
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()

    pages = data.get("query", {}).get("pages", {})
    for page_id, page_info in pages.items():
        if "extract" in page_info:
            return page_info["extract"]

    return "No text available."

def clean_filename(title):
    """Removes invalid characters so the title can safely be used as a filename."""
    # Keep only letters, numbers, and spaces, then replace spaces with underscores
    clean = re.sub(r'[^a-zA-Z0-9 ]', '', title)
    return clean.replace(' ', '_')

if __name__ == "__main__":
    print("Fetching top 10 AI company titles...\n")
    company_titles = get_ai_companies(10)

    zip_filename = "AI_Companies_Articles.zip"

    if company_titles:
        print(f"Creating {zip_filename}...\n")

        # Open a zip file to write into
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for i, title in enumerate(company_titles, start=1):
                print(f"[{i}/10] Crawling and saving: {title}")

                # Fetch text
                article_text = get_article_text(title)

                # Create a safe filename (e.g. "OpenAI.txt")
                filename = f"{clean_filename(title)}.txt"

                # Write the text directly into a file inside the zip archive
                zipf.writestr(filename, article_text)

        print(f"\nSuccess! All 10 articles have been saved to '{zip_filename}' in your current directory.")
    else:
        print("No companies found.")

Fetching top 10 AI company titles...

Creating AI_Companies_Articles.zip...

[1/10] Crawling and saving: List of artificial intelligence companies
[2/10] Crawling and saving: 01.AI
[3/10] Crawling and saving: 1X Technologies
[4/10] Crawling and saving: 4Paradigm
[5/10] Crawling and saving: AAI Labs
[6/10] Crawling and saving: Adaptive ML
[7/10] Crawling and saving: Advanced Machine Intelligence Labs
[8/10] Crawling and saving: Agility Robotics
[9/10] Crawling and saving: AI21 Labs
[10/10] Crawling and saving: Aikido Security


HTTPError: 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&prop=extracts&titles=Aikido+Security&format=json&explaintext=True&exintro=False

In [25]:
import zipfile
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Unzip the articles (if you haven't already)
zip_filename = "AI_Companies_Articles.zip"
extract_folder = "ai_company_texts"

if not os.path.exists(extract_folder):
    print(f"Extracting {zip_filename} to folder '{extract_folder}'...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)

# 2. Use DirectoryLoader to load ALL text files in the folder
print("\nLoading documents with LangChain...")
# The glob="**/*.txt" ensures it only picks up text files
loader = DirectoryLoader(extract_folder, glob="**/*.txt", loader_cls=TextLoader)
docs = loader.load()

print(f"Successfully loaded {len(docs)} full articles.")

# 3. Split the loaded documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=24)
documents = text_splitter.split_documents(documents=docs)

print(f"Split the articles into {len(documents)} readable chunks.")

# (Optional) Print the first chunk just to verify it worked
if documents:
    print("\n--- Sample Chunk ---")
    print(f"Source: {documents[0].metadata['source']}")
    print(f"Text: {documents[0].page_content}")

Extracting AI_Companies_Articles.zip to folder 'ai_company_texts'...

Loading documents with LangChain...
Successfully loaded 9 full articles.
Split the articles into 14 readable chunks.

--- Sample Chunk ---
Source: ai_company_texts/1X_Technologies.txt
Text: 1X Technologies is a Norwegian-American robotics and artificial intelligence company developing general-purpose humanoid robots for home environments. The company is headquartered in Palo Alto, California, with main manufacturing operations in


# Build graph using openai

In [ ]:
# from langchain_ollama import ChatOllama

# llm = ChatOllama(
#     model="llama3.1",
#     temperature=0,
#     base_url="http://localhost:11434",
#     format="json"
# )

import os
os.environ["OPENAI_API_KEY"] = 

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [47]:
llm_transformer = LLMGraphTransformer(llm=llm)

graph_documents = llm_transformer.convert_to_graph_documents(documents)


In [51]:
graph_documents[0]

GraphDocument(nodes=[Node(id='1X Technologies', type='Company', properties={}), Node(id='Norwegian-American', type='Nationality', properties={}), Node(id='Palo Alto', type='Location', properties={}), Node(id='California', type='Location', properties={}), Node(id='Home Environments', type='Environment', properties={}), Node(id='Humanoid Robots', type='Technology', properties={}), Node(id='Artificial Intelligence', type='Technology', properties={}), Node(id='Robotics', type='Technology', properties={})], relationships=[Relationship(source=Node(id='1X Technologies', type='Company', properties={}), target=Node(id='Norwegian-American', type='Nationality', properties={}), type='HAS_NATIONALITY', properties={}), Relationship(source=Node(id='1X Technologies', type='Company', properties={}), target=Node(id='Palo Alto', type='Location', properties={}), type='HEADQUARTERED_IN', properties={}), Relationship(source=Node(id='Palo Alto', type='Location', properties={}), target=Node(id='California', t

In [49]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [58]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Neo4jVector

embeddings = OpenAIEmbeddings(
    # model="text-embedding-3-large"  # best quality
    model="text-embedding-3-small"
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding_new",
    index_name="vector_openai",
    database="b44e89f3"
)

vector_retriever = vector_index.as_retriever()

In [59]:
# driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id`
    FOR (n:__Entity__)
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session() as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    pass

# Close the driver connection
driver.close()

In [60]:

class Entities(BaseModel):
    """Identifying information about entities."""

    names: list[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)


entity_chain = llm.with_structured_output(Entities)

# Build GT for eval

In [72]:
# ── QUESTION LIST (ordered easy → hard) ────────────────────────────

TEST_QUESTIONS: list[str] = [

    # ── EASY: single-document, direct attribute lookup ───────────────
    # Q01
    "Where is 01.AI headquartered?",

    # Q02
    "What does AI21 Labs specialize in?",

    # Q03
    "When was AAI Labs founded?",

    # Q04
    "What is the name of Agility Robotics' humanoid robot?",

    # Q05
    "On which stock exchange is 4Paradigm publicly listed?",

    # ── MEDIUM: single-doc, multi-sentence synthesis ─────────────────
    # Q06
    "What is Adaptive ML's primary technical focus and what kind of models does it help customize?",

    # Q07
    "Where does 1X Technologies have its manufacturing operations, and where is it headquartered?",

    # Q08
    "How much did Advanced Machine Intelligence Labs raise in its seed funding round, and who were the investors?",

    # Q09
    "What university did Agility Robotics originate from, and in what year was it founded?",

    # Q10
    "What specific AI capabilities does Advanced Machine Intelligence Labs aim to develop in its AI systems?",

    # ── MEDIUM-HARD: cross-document, shared property linking ──────────
    # Q11
    "Which companies in the dataset are developing humanoid robots?",

    # Q12
    "Which AI companies in the dataset have a presence in Paris, France?",

    # Q13
    "Which companies in the dataset focus on open-source AI or open-source large language models?",

    # Q14
    "Which companies in the dataset are Chinese or primarily operate in China?",

    # Q15
    "Which companies in the dataset serve government or public sector clients?",

    # ── HARD: multi-hop reasoning, answer lives in graph edges ────────
    # Q16 — 2-hop: Advanced Machine Intelligence Labs → Yann LeCun → Meta
    "What major technology corporation did the co-founder of Advanced Machine Intelligence Labs previously work for, and in what role?",

    # Q17 — 2-hop cross-entity: shared city Paris → Adaptive ML + Advanced Machine Intelligence Labs
    "Name all the AI companies in the dataset that have offices or operations in Paris, and describe what each of them does.",

    # Q18 — 2-hop: Advanced Machine Intelligence Labs → investors → Nvidia (known for AI hardware)
    "Which of the investors in Advanced Machine Intelligence Labs' seed round is also a dominant supplier of AI hardware, and what does that company make?",

    # Q19 — 3-hop: Agility Robotics (university spin-off) vs Advanced Machine Intelligence Labs (corporate-scientist spin-off), compare origin paths
    "Compare how Agility Robotics and Advanced Machine Intelligence Labs were each created: trace their founding origins back to their source institution or person.",

    # Q20 — 3-hop: 1X Technologies locations → Norwegian-American → Europe+US presence; Adaptive ML → NY+Paris → same pattern; Advanced Machine Intelligence Labs → Paris + US investors; find the common thread
    "Which companies in the dataset operate across both Europe and North America, and what evidence in the articles supports this for each?",
]

# ── GROUND TRUTHS ───────────────────────────────────────────────────

GROUND_TRUTHS: dict[str, str | None] = {

    # Q01 — easy, direct
    "Where is 01.AI headquartered?":
        "01.AI is based in Beijing, China.",

    # Q02 — easy, direct
    "What does AI21 Labs specialize in?":
        "AI21 Labs is an Israeli company specializing in Natural Language Processing (NLP). "
        "It develops AI systems that can understand and generate natural language.",

    # Q03 — easy, direct
    "When was AAI Labs founded?":
        "AAI Labs was founded in 2018.",

    # Q04 — easy, direct
    "What is the name of Agility Robotics' humanoid robot?":
        "Agility Robotics' humanoid robot is called Digit.",

    # Q05 — easy, direct
    "On which stock exchange is 4Paradigm publicly listed?":
        "4Paradigm is publicly listed in Hong Kong.",

    # Q06 — medium, multi-sentence
    "What is Adaptive ML's primary technical focus and what kind of models does it help customize?":
        "Adaptive ML focuses on reinforcement learning, specifically 'RLOps'. "
        "It provides tools that allow organizations to customize and operate "
        "open-source large language models for specific applications.",

    # Q07 — medium, multi-sentence
    "Where does 1X Technologies have its manufacturing operations, and where is it headquartered?":
        "1X Technologies is headquartered in Palo Alto, California. "
        "It has main manufacturing operations in Hayward, California, and "
        "additional manufacturing in Moss, Norway.",

    # Q08 — medium, multi-sentence
    "How much did Advanced Machine Intelligence Labs raise in its seed funding round, and who were the investors?":
        "Advanced Machine Intelligence Labs raised $1.03 billion in a seed funding round. "
        "The investors were Bezos Expeditions, Nvidia, and Samsung Electronics.",

    # Q09 — medium, multi-sentence
    "What university did Agility Robotics originate from, and in what year was it founded?":
        "Agility Robotics was founded in 2015 as a spin-off from Oregon State University.",

    # Q10 — medium, multi-sentence
    "What specific AI capabilities does Advanced Machine Intelligence Labs aim to develop in its AI systems?":
        "Advanced Machine Intelligence Labs focuses on developing AI systems that "
        "understand the physical world, maintain long-term memory, and strategize "
        "complicated tasks, while remaining secure and easily manageable.",

    # Q11 — medium-hard, cross-document
    "Which companies in the dataset are developing humanoid robots?":
        "Two companies are developing humanoid robots: "
        "1X Technologies, which develops general-purpose humanoid robots for home environments, "
        "and Agility Robotics, which builds the humanoid robot Digit for automation solutions.",

    # Q12 — medium-hard, cross-document
    "Which AI companies in the dataset have a presence in Paris, France?":
        "Two companies have a presence in Paris: "
        "Adaptive ML, which is based in both New York and Paris, and "
        "Advanced Machine Intelligence Labs, which is headquartered in Paris.",

    # Q13 — medium-hard, cross-document
    "Which companies in the dataset focus on open-source AI or open-source large language models?":
        "01.AI focuses on developing open-source products. "
        "Adaptive ML provides tools for customizing and operating open-source "
        "large language models.",

    # Q14 — medium-hard, cross-document
    "Which companies in the dataset are Chinese or primarily operate in China?":
        "01.AI is based in Beijing, China and develops open-source AI products. "
        "4Paradigm (doing business as Phancy Group) is a Chinese company publicly "
        "listed in Hong Kong that provides AI tools and applications in China.",

    # Q15 — medium-hard, cross-document
    "Which companies in the dataset serve government or public sector clients?":
        "AAI Labs serves both public sector institutions and private enterprises. "
        "It has participated in government technology (GovTech) initiatives and "
        "European innovation collaboration programmes.",

    # Q16 — HARD: 2-hop (company → founder → prior employer)
    "What major technology corporation did the co-founder of Advanced Machine Intelligence Labs previously work for, and in what role?":
        "Advanced Machine Intelligence Labs was co-founded by Yann LeCun. "
        "Yann LeCun previously served as the Chief AI Scientist at Meta "
        "(formerly Facebook). This is a two-hop connection: "
        "Advanced Machine Intelligence Labs → co-founded by Yann LeCun → former Chief AI Scientist at Meta.",

    # Q17 — HARD: 2-hop cross-entity (Paris presence → find all companies → describe each)
    "Name all the AI companies in the dataset that have offices or operations in Paris, and describe what each of them does.":
        "Two companies have Paris operations: "
        "(1) Adaptive ML, based in New York and Paris, focuses on reinforcement learning "
        "(RLOps) and helps organizations customize open-source large language models. "
        "(2) Advanced Machine Intelligence Labs, a Paris-based startup co-founded by Yann LeCun, "
        "develops AI systems that understand the physical world, maintain long-term memory, "
        "and strategize complex tasks. It raised $1.03B from Bezos Expeditions, Nvidia, "
        "and Samsung Electronics.",

    # Q18 — HARD: 2-hop (Advanced Machine Intelligence Labs → investor Nvidia → Nvidia's hardware role)
    "Which of the investors in Advanced Machine Intelligence Labs' seed round is also a dominant supplier of AI hardware, and what does that company make?":
        "Nvidia invested in Advanced Machine Intelligence Labs' $1.03 billion seed round. "
        "Nvidia is also the dominant supplier of AI hardware, best known for its "
        "graphics processing units (GPUs) that are widely used to train and run "
        "large AI models. The link is: Advanced Machine Intelligence Labs "
        "→ funded by Nvidia → Nvidia is a leading AI hardware manufacturer.",

    # Q19 — HARD: 3-hop compare two founding origin paths
    "Compare how Agility Robotics and Advanced Machine Intelligence Labs were each created: trace their founding origins back to their source institution or person.":
        "Agility Robotics followed an academic spin-off path: it was founded in 2015 "
        "as a spin-off from Oregon State University, transferring university robotics "
        "research into a commercial company. "
        "Advanced Machine Intelligence Labs followed a corporate-researcher-to-startup path: "
        "it was co-founded by Yann LeCun, who left his role as Chief AI Scientist at Meta "
        "to start the company in 2026. "
        "Both paths trace back to an institution—one academic (Oregon State), one corporate (Meta)—"
        "but represent different technology transfer mechanisms.",

    # Q20 — HARD: 3-hop, find all transatlantic companies + evidence
    "Which companies in the dataset operate across both Europe and North America, and what evidence in the articles supports this for each?":
        "Three companies span both Europe and North America: "
        "(1) 1X Technologies: headquartered in Palo Alto, California (North America), "
        "with manufacturing in Hayward, CA and additional manufacturing in Moss, Norway (Europe). "
        "(2) Adaptive ML: based in New York, United States (North America) and Paris, France (Europe). "
        "(3) Advanced Machine Intelligence Labs: headquartered in Paris, France (Europe) "
        "and funded by American investors Bezos Expeditions and Nvidia, indicating strong "
        "transatlantic financial ties. "
        "AAI Labs (Lithuania) and AI21 Labs (Israel) are European/Middle Eastern but "
        "lack explicit North American presence in the source documents.",
}


# ── METADATA (optional, for analysis / reporting) ──────────────────

QUESTION_METADATA: list[dict] = [
    # Q01–Q05: EASY
    {"id": "Q01", "question": TEST_QUESTIONS[0],  "difficulty": "easy",        "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q02", "question": TEST_QUESTIONS[1],  "difficulty": "easy",        "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q03", "question": TEST_QUESTIONS[2],  "difficulty": "easy",        "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q04", "question": TEST_QUESTIONS[3],  "difficulty": "easy",        "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q05", "question": TEST_QUESTIONS[4],  "difficulty": "easy",        "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    # Q06–Q10: MEDIUM
    {"id": "Q06", "question": TEST_QUESTIONS[5],  "difficulty": "medium",      "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q07", "question": TEST_QUESTIONS[6],  "difficulty": "medium",      "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q08", "question": TEST_QUESTIONS[7],  "difficulty": "medium",      "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q09", "question": TEST_QUESTIONS[8],  "difficulty": "medium",      "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    {"id": "Q10", "question": TEST_QUESTIONS[9],  "difficulty": "medium",      "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    # Q11–Q15: MEDIUM-HARD
    {"id": "Q11", "question": TEST_QUESTIONS[10], "difficulty": "medium-hard", "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q12", "question": TEST_QUESTIONS[11], "difficulty": "medium-hard", "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q13", "question": TEST_QUESTIONS[12], "difficulty": "medium-hard", "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q14", "question": TEST_QUESTIONS[13], "difficulty": "medium-hard", "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q15", "question": TEST_QUESTIONS[14], "difficulty": "medium-hard", "hops": 1, "docs_needed": 1, "expected_winner": "tie"},
    # Q16–Q20: HARD
    {"id": "Q16", "question": TEST_QUESTIONS[15], "difficulty": "hard",        "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q17", "question": TEST_QUESTIONS[16], "difficulty": "hard",        "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q18", "question": TEST_QUESTIONS[17], "difficulty": "hard",        "hops": 2, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q19", "question": TEST_QUESTIONS[18], "difficulty": "hard",        "hops": 3, "docs_needed": 2, "expected_winner": "GraphRAG"},
    {"id": "Q20", "question": TEST_QUESTIONS[19], "difficulty": "hard",        "hops": 3, "docs_needed": 3, "expected_winner": "GraphRAG"},
]


# Inference

In [63]:
entity_chain.invoke("What is the name of Agility Robotics' humanoid robot?")

Entities(names=['Agility Robotics', 'Cassie'])

In [64]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


# Fulltext index query
def graph_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke(question)
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": entity},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [65]:
print(graph_retriever("What is the name of Agility Robotics' humanoid robot?"))

Agility Robotics, Inc. - PROVIDES -> Humanoid Robot
Agility Robotics, Inc. - SPIN_OFF -> Oregon State University
1X Technologies - DEVELOPS -> Robotics


In [66]:
def full_retriever(question: str):
    graph_data = graph_retriever(question)
    vector_data = [el.page_content for el in vector_retriever.invoke(question)]
    final_data = f"""Graph data:
{graph_data}
vector data:
{"#Document ". join(vector_data)}
    """
    return final_data

In [67]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
        {
            "context": full_retriever,
            "question": RunnablePassthrough(),
        }
    | prompt
    | llm
    | StrOutputParser()
)

In [68]:
chain.invoke(input="What is Adaptive ML's primary technical focus and what kind of models does it help customize?")

"Adaptive ML's primary technical focus is on reinforcement learning (RLOps). The company helps customize and operate open-source large language models for specific applications."

# Eval

In [ ]:
# -*- coding: utf-8 -*-
"""
======================================================================
  EVALUATION: Flat RAG vs GraphRAG
  Model     : gpt-4o-mini  (OpenAI)
  Embeddings: text-embedding-3-small (OpenAI)
======================================================================
  FIXES vs previous version:
    [1] KeyError on JUDGE_SYSTEM  — curly braces inside the JSON
        example were parsed as LangChain template variables.
        Fixed by escaping all literal braces as {{ and }}
    [2] Migrated LLM  : ChatOllama  → ChatOpenAI (gpt-4o-mini)
    [3] Migrated Embed: OllamaEmbed → OpenAIEmbeddings (text-embedding-3-small)
    [4] Entity extraction uses with_structured_output() on ChatOpenAI
        (native function-calling, no json format hack needed)
    [5] Judge uses model_kwargs response_format json_object for
        guaranteed valid JSON back from the model
    [6] try/except wraps each method call so one failure doesn't
        crash the entire evaluation run
======================================================================
"""

# ── 0. DEPENDENCIES ──────────────────────────────────────────────────
# pip install langchain langchain-community langchain-openai
#             langchain-experimental neo4j tiktoken
#             python-dotenv pydantic tabulate colorama

import os, time, json, statistics
from dataclasses import dataclass, field, asdict
from typing import Optional

from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False

try:
    from colorama import Fore, Style, init as colorama_init
    colorama_init(autoreset=True)
    HAS_COLOR = True
except ImportError:
    HAS_COLOR = False

# ── 1. CONFIG ─────────────────────────────────────────────────────────
NEO4J_URI      = os.environ.get("NEO4J_URI",      "neo4j+s://b44e89f3.databases.neo4j.io")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME",  "")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD",  "")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE",  "")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY",  "")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

# ── 2. MODELS ─────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

# Judge gets response_format=json_object so output is always valid JSON
llm_json = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
    model_kwargs={"response_format": {"type": "json_object"}},
)

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

# ── 3. GRAPH & VECTOR SETUP ───────────────────────────────────────────
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding_new",
    index_name="vector_openai",
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

vector_retriever = vector_index.as_retriever(search_kwargs={"k": 4})

# ── 4. ENTITY EXTRACTION (GraphRAG) ──────────────────────────────────
class Entities(BaseModel):
    """Named entities extracted from the question."""
    names: list[str] = Field(
        ...,
        description="All person, organization, or business entities in the text.",
    )

# ChatOpenAI supports with_structured_output via native function-calling
entity_chain = llm.with_structured_output(Entities)

def graph_retriever(question: str) -> str:
    """Traverse the knowledge graph neighbourhood of extracted entities."""
    result = ""
    try:
        entities = entity_chain.invoke(question)
        for entity in entities.names:
            words = [w for w in remove_lucene_chars(entity).split() if w]
            if not words:
                continue
            query = " AND ".join([f"{w}~2" for w in words])
            response = graph.query(
                """
                CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit:2})
                YIELD node, score
                CALL {
                  WITH node
                  MATCH (node)-[r:!MENTIONS]->(neighbor)
                  RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
                  UNION ALL
                  WITH node
                  MATCH (node)<-[r:!MENTIONS]-(neighbor)
                  RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
                }
                RETURN output LIMIT 50
                """,
                {"query": query},
            )
            result += "\n".join([el["output"] for el in response])
    except Exception as e:
        result += f"[Graph retrieval error: {e}]"
    return result

# ── 5. RETRIEVER FUNCTIONS ────────────────────────────────────────────
def flat_rag_context(question: str) -> str:
    """Flat RAG: vector similarity only."""
    docs = vector_retriever.invoke(question)
    return "#Document ".join([d.page_content for d in docs])

def graph_rag_context(question: str) -> str:
    """GraphRAG: graph traversal + vector hybrid."""
    graph_data  = graph_retriever(question)
    vector_data = [d.page_content for d in vector_retriever.invoke(question)]
    return (
        f"Graph data:\n{graph_data}\n\n"
        f"Vector data:\n{'#Document '.join(vector_data)}"
    )

# ── 6. QA CHAINS ──────────────────────────────────────────────────────
QA_TEMPLATE = """Answer the question based ONLY on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

qa_prompt = ChatPromptTemplate.from_template(QA_TEMPLATE)

def build_chain(retriever_fn):
    return (
        {"context": retriever_fn, "question": RunnablePassthrough()}
        | qa_prompt
        | llm
        | StrOutputParser()
    )

flat_chain  = build_chain(flat_rag_context)
graph_chain = build_chain(graph_rag_context)

# ── 7. LLM-AS-JUDGE ───────────────────────────────────────────────────
# FIX: literal {{ }} prevents LangChain from treating them as variables.
JUDGE_SYSTEM = """\
You are an expert evaluator of RAG (Retrieval Augmented Generation) systems.
You will receive a question, a retrieved context, and a generated answer.
Score each dimension from 0.0 to 1.0 (two decimal places).

Return ONLY a valid JSON object with exactly these keys (no markdown fences):
{{
  "faithfulness": <0.0-1.0>,
  "answer_relevancy": <0.0-1.0>,
  "context_recall": <0.0-1.0>,
  "completeness": <0.0-1.0>,
  "reasoning": "<one-sentence justification>"
}}

Definitions:
- faithfulness:      Is the answer grounded in the context (no hallucination)?
- answer_relevancy:  Does the answer directly address the question?
- context_recall:    Does the context contain the information needed to answer?
- completeness:      Is the answer thorough and not missing key details?"""

JUDGE_HUMAN = """\
Question: {question}
Context:  {context}
Answer:   {answer}
{ground_truth_block}
Score:"""

judge_prompt = ChatPromptTemplate.from_messages([
    ("system", JUDGE_SYSTEM),
    ("human",  JUDGE_HUMAN),
])
judge_chain = judge_prompt | llm_json | StrOutputParser()

# ── 8. DATA CLASSES ───────────────────────────────────────────────────
@dataclass
class Scores:
    faithfulness:     float = 0.0
    answer_relevancy: float = 0.0
    context_recall:   float = 0.0
    completeness:     float = 0.0
    reasoning:        str   = ""

def judge(question: str, context: str, answer: str,
          ground_truth: Optional[str] = None) -> Scores:
    gt_block = f"Ground truth: {ground_truth}" if ground_truth else ""
    try:
        raw = judge_chain.invoke({
            "question":           question,
            "context":            context[:3000],
            "answer":             answer,
            "ground_truth_block": gt_block,
        })
        raw = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        data = json.loads(raw)
        return Scores(
            faithfulness     = float(data.get("faithfulness",     0)),
            answer_relevancy = float(data.get("answer_relevancy", 0)),
            context_recall   = float(data.get("context_recall",   0)),
            completeness     = float(data.get("completeness",     0)),
            reasoning        = str(data.get("reasoning", "")),
        )
    except Exception as e:
        return Scores(reasoning=f"[Judge error: {e}]")

@dataclass
class EvalResult:
    question:    str
    method:      str        # "Flat RAG" | "GraphRAG"
    context:     str   = ""
    answer:      str   = ""
    latency_s:   float = 0.0
    context_len: int   = 0
    scores:      Scores = field(default_factory=Scores)

    @property
    def avg_score(self) -> float:
        s = self.scores
        return round(statistics.mean([
            s.faithfulness, s.answer_relevancy,
            s.context_recall, s.completeness,
        ]), 3)

# ── 9. EVALUATION LOOP ────────────────────────────────────────────────
def _c(text, color=""):
    return f"{color}{text}{Style.RESET_ALL}" if HAS_COLOR else text

def run_evaluation() -> list[EvalResult]:
    results: list[EvalResult] = []
    total = len(TEST_QUESTIONS)

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\n{'='*68}")
        print(_c(f"  [{i}/{total}] {question}", Fore.CYAN if HAS_COLOR else ""))
        print(f"{'='*68}")
        gt = GROUND_TRUTHS.get(question)

        for method_name, chain, ctx_fn in [
            ("Flat RAG", flat_chain,  flat_rag_context),
            ("GraphRAG", graph_chain, graph_rag_context),
        ]:
            print(_c(f"\n  ▶ {method_name}", Fore.YELLOW if HAS_COLOR else ""))
            try:
                t0      = time.perf_counter()
                context = ctx_fn(question)
                answer  = chain.invoke(question)
                latency = round(time.perf_counter() - t0, 3)

                print(f"    Latency : {latency}s")
                print(f"    Answer  : {answer[:120]}{'…' if len(answer) > 120 else ''}")

                scores = judge(question, context, answer, gt)
                print(f"    Scores  → Faith={scores.faithfulness:.2f} | "
                      f"Rel={scores.answer_relevancy:.2f} | "
                      f"Recall={scores.context_recall:.2f} | "
                      f"Complete={scores.completeness:.2f}")
                print(f"    Reason  : {scores.reasoning}")

                results.append(EvalResult(
                    question    = question,
                    method      = method_name,
                    context     = context,
                    answer      = answer,
                    latency_s   = latency,
                    context_len = len(context),
                    scores      = scores,
                ))

            except Exception as e:
                print(_c(f"    [ERROR] {e}", Fore.RED if HAS_COLOR else ""))
                results.append(EvalResult(
                    question = question,
                    method   = method_name,
                    scores   = Scores(reasoning=f"[Runtime error: {e}]"),
                ))

    return results

# ── 10. REPORTING ─────────────────────────────────────────────────────
def aggregate(results: list[EvalResult], method: str) -> dict:
    subset = [r for r in results if r.method == method]
    if not subset:
        return {}
    def mean(attr):
        return round(statistics.mean([getattr(r.scores, attr) for r in subset]), 3)
    return {
        "method":           method,
        "n_questions":      len(subset),
        "avg_faithfulness": mean("faithfulness"),
        "avg_relevancy":    mean("answer_relevancy"),
        "avg_recall":       mean("context_recall"),
        "avg_completeness": mean("completeness"),
        "avg_overall":      round(statistics.mean([r.avg_score  for r in subset]), 3),
        "avg_latency_s":    round(statistics.mean([r.latency_s  for r in subset]), 3),
        "avg_ctx_len":      round(statistics.mean([r.context_len for r in subset])),
    }

def print_report(results: list[EvalResult]):
    print("\n\n" + "═"*68)
    print("  EVALUATION REPORT — Flat RAG vs GraphRAG")
    print(f"  LLM: {LLM_MODEL}   Embeddings: {EMBED_MODEL}")
    print("═"*68)

    flat_agg  = aggregate(results, "Flat RAG")
    graph_agg = aggregate(results, "GraphRAG")

    metric_keys = [
        ("avg_faithfulness",  "Faithfulness"),
        ("avg_relevancy",     "Answer Relevancy"),
        ("avg_recall",        "Context Recall"),
        ("avg_completeness",  "Completeness"),
        ("avg_overall",       "── OVERALL SCORE ──"),
        ("avg_latency_s",     "Latency (s)"),
        ("avg_ctx_len",       "Context Length (chars)"),
    ]

    rows = []
    for key, label in metric_keys:
        fv = flat_agg.get(key, "-")
        gv = graph_agg.get(key, "-")
        if isinstance(fv, (int, float)) and isinstance(gv, (int, float)):
            if key == "avg_latency_s":
                winner = "◀ GraphRAG" if gv < fv else ("Flat RAG ▶" if fv < gv else "tie")
            else:
                winner = "◀ GraphRAG" if gv > fv else ("Flat RAG ▶" if fv > gv else "tie")
        else:
            winner = ""
        rows.append([label, fv, gv, winner])

    if HAS_TABULATE:
        print(tabulate(rows, headers=["Metric","Flat RAG","GraphRAG","Winner"],
                       tablefmt="fancy_grid", floatfmt=".3f"))
    else:
        hdr = f"{'Metric':<26} {'Flat RAG':>10} {'GraphRAG':>10}  Winner"
        print(hdr); print("-"*len(hdr))
        for r in rows:
            print(f"{r[0]:<26} {str(r[1]):>10} {str(r[2]):>10}  {r[3]}")

    print("\n\n  PER-QUESTION BREAKDOWN")
    print("─"*68)
    pq = []
    for q in TEST_QUESTIONS:
        fr = next((r for r in results if r.question == q and r.method == "Flat RAG"),  None)
        gr = next((r for r in results if r.question == q and r.method == "GraphRAG"), None)
        if fr and gr:
            pq.append([
                (q[:44]+"…") if len(q) > 45 else q,
                fr.avg_score, gr.avg_score,
                "◀ G" if gr.avg_score > fr.avg_score else
                ("F ▶" if fr.avg_score > gr.avg_score else "tie"),
            ])

    if HAS_TABULATE:
        print(tabulate(pq, headers=["Question","Flat RAG","GraphRAG","Win"],
                       tablefmt="simple", floatfmt=".3f"))
    else:
        for r in pq:
            print(f"  {r[0]:<46} F={r[1]:.3f}  G={r[2]:.3f}  {r[3]}")

    output = {
        "config": {"llm": LLM_MODEL, "embeddings": EMBED_MODEL},
        "summary": {"flat_rag": flat_agg, "graphrag": graph_agg},
        "per_question": [
            {"question": r.question, "method": r.method, "answer": r.answer,
             "latency_s": r.latency_s, "scores": asdict(r.scores),
             "avg_score": r.avg_score}
            for r in results
        ],
    }
    with open("eval_results.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"\n  ✔ Results saved → eval_results.json")
    print("═"*68 + "\n")

# ── 11. ENTRY POINT ───────────────────────────────────────────────────
if __name__ == "__main__":
    if not OPENAI_API_KEY:
        raise EnvironmentError("OPENAI_API_KEY not set. Add to .env or export it.")

    print("\n" + "═"*68)
    print("  RAG vs GraphRAG — Evaluation Suite")
    print(f"  LLM       : {LLM_MODEL}")
    print(f"  Embeddings: {EMBED_MODEL}")
    print(f"  Questions : {len(TEST_QUESTIONS)}")
    print("═"*68)

    results = run_evaluation()
    print_report(results)


════════════════════════════════════════════════════════════════════
  RAG vs GraphRAG — Evaluation Suite
  LLM       : gpt-4o-mini
  Embeddings: text-embedding-3-small
  Questions : 20
════════════════════════════════════════════════════════════════════

  [1/20] Where is 01.AI headquartered?

  ▶ Flat RAG


    Latency : 1.952s
    Answer  : 01.AI is headquartered in Beijing, China.
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=1.00
    Reason  : The answer accurately reflects the information provided in the context and directly addresses the question.

  ▶ GraphRAG


    Latency : 4.571s
    Answer  : 01.AI is headquartered in Beijing, China.
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=1.00
    Reason  : The answer accurately reflects the information provided in the context and directly addresses the question.

  [2/20] What does AI21 Labs specialize in?

  ▶ Flat RAG


    Latency : 2.431s
    Answer  : AI21 Labs specializes in Natural Language Processing (NLP) and develops AI systems that can understand and generate natu…
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=1.00
    Reason  : The answer accurately reflects the information provided in the context without any omissions or inaccuracies.

  ▶ GraphRAG


    Latency : 5.252s
    Answer  : AI21 Labs specializes in Natural Language Processing (NLP).
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=0.80
    Reason  : The answer accurately reflects the specialization of AI21 Labs as stated in the context, but it omits the detail about developing AI systems that can understand and generate natural language.

  [3/20] When was AAI Labs founded?

  ▶ Flat RAG


...

    Latency : 5.915s
    Answer  : The investor in Advanced Machine Intelligence Labs' seed round that is also a dominant supplier of AI hardware is Nvidia…
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=1.00
    Reason  : The answer accurately identifies Nvidia as the investor and describes its product, fully aligning with the context provided.

  [19/20] Compare how Agility Robotics and Advanced Machine Intelligence Labs were each created: trace their founding origins back to their source institution or person.

  ▶ Flat RAG


    Latency : 3.143s
    Answer  : Agility Robotics, Inc. was founded in 2015 as a spin-off from Oregon State University, indicating its origins in academi…
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=0.90
    Reason  : The answer accurately reflects the founding origins of both companies and addresses the question directly, but it could include more detail about the technology transfer mechanisms.

  ▶ GraphRAG


    Latency : 10.969s
    Answer  : Agility Robotics, Inc. was founded in 2015 as a spin-off from Oregon State University, indicating its origins in academi…
    Scores  → Faith=1.00 | Rel=1.00 | Recall=1.00 | Complete=0.90
    Reason  : The answer accurately reflects the founding origins of both companies based on the context, but it could include more detail about the technology transfer mechanisms for completeness.

  [20/20] Which companies in the dataset operate across both Europe and North America, and what evidence in the articles supports this for each?

  ▶ Flat RAG


    Latency : 3.198s
    Answer  : The company that operates across both Europe and North America is Adaptive ML. The evidence supporting this is that Adap…
    Scores  → Faith=0.67 | Rel=1.00 | Recall=0.33 | Complete=0.33
    Reason  : The answer correctly identifies Adaptive ML as operating in both regions but fails to mention the other two companies that also meet the criteria, leading to incomplete context recall and completeness.

  ▶ GraphRAG


    Latency : 11.0s
    Answer  : The company that operates across both Europe and North America is Adaptive ML. The evidence supporting this is that Adap…
    Scores  → Faith=1.00 | Rel=1.00 | Recall=0.33 | Complete=0.25
    Reason  : The answer correctly identifies Adaptive ML as operating in both regions but fails to mention the other two companies that also meet the criteria.


════════════════════════════════════════════════════════════════════
  EVALUATION REPORT — Flat RAG vs GraphRAG
  LLM: gpt-4o-mini   Embeddings: text-embedding-3-small
════════════════════════════════════════════════════════════════════
╒════════════════════════╤════════════╤════════════╤════════════╕
│ Metric                 │   Flat RAG │   GraphRAG │ Winner     │
╞════════════════════════╪════════════╪════════════╪════════════╡
│ Faithfulness           │      0.874 │      0.950 │ ◀ GraphRAG │
├────────────────────────┼────────────┼────────────┼────────────┤
│ Answer Relevancy       │      0.990 │      1.0

In [74]:
# ── GRAPHRAG BUILD COST TRACKER ───────────────────────────────────────
import time
import tiktoken
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI

# ── Pricing (USD per 1M tokens) — update if OpenAI changes pricing ──
PRICING = {
    "gpt-4o-mini": {"input": 0.150, "output": 0.600},
    "gpt-4o":      {"input": 5.000, "output": 15.000},
    "gpt-4-turbo": {"input": 10.00, "output": 30.000},
}

MODEL = "gpt-4o-mini"  # must match the llm you pass to LLMGraphTransformer

# ── Token counter using tiktoken ─────────────────────────────────────
enc = tiktoken.encoding_for_model(MODEL)

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

# ── Instrumented LLM wrapper ─────────────────────────────────────────
class TokenTrackingLLM:
    """Wraps ChatOpenAI and accumulates token counts via callbacks."""

    def __init__(self, model: str = MODEL):
        self.model = model
        self.total_input_tokens  = 0
        self.total_output_tokens = 0
        self._llm = ChatOpenAI(
            model=model,
            temperature=0,
            # stream=False ensures usage metadata is returned
        )

    def invoke(self, *args, **kwargs):
        response = self._llm.invoke(*args, **kwargs)
        # LangChain ≥ 0.1.x exposes usage_metadata on AIMessage
        if hasattr(response, "usage_metadata") and response.usage_metadata:
            self.total_input_tokens  += response.usage_metadata.get("input_tokens", 0)
            self.total_output_tokens += response.usage_metadata.get("output_tokens", 0)
        else:
            # Fallback: estimate from text
            prompt_text = str(args[0]) if args else ""
            self.total_input_tokens  += count_tokens(prompt_text)
            self.total_output_tokens += count_tokens(response.content)
        return response

    # LLMGraphTransformer calls the underlying LLM directly via the chain,
    # so we expose __getattr__ to proxy everything else to the real LLM.
    def __getattr__(self, name):
        return getattr(self._llm, name)

# ── Build the graph with cost tracking ───────────────────────────────
print("=" * 60)
print("  GraphRAG — Knowledge Graph Build (Cost Tracker)")
print("=" * 60)

tracking_llm = TokenTrackingLLM(model=MODEL)
llm_transformer = LLMGraphTransformer(llm=tracking_llm._llm)

# ── START timing ─────────────────────────────────────────────────────
t_start = time.perf_counter()

graph_documents = llm_transformer.convert_to_graph_documents(documents)

t_end   = time.perf_counter()
elapsed = t_end - t_start

# ── Compute costs ─────────────────────────────────────────────────────
# Since LLMGraphTransformer uses its own internal chain we estimate
# tokens from the document corpus + graph output as a reliable fallback.

total_input_chars  = sum(len(d.page_content) for d in documents)
total_output_chars = sum(
    len(str(gd.nodes)) + len(str(gd.relationships))
    for gd in graph_documents
)

# tiktoken estimate (accurate for OpenAI models)
est_input_tokens  = count_tokens(" ".join(d.page_content for d in documents))
est_output_tokens = count_tokens(
    " ".join(
        str(gd.nodes) + str(gd.relationships)
        for gd in graph_documents
    )
)

prices = PRICING[MODEL]
input_cost  = (est_input_tokens  / 1_000_000) * prices["input"]
output_cost = (est_output_tokens / 1_000_000) * prices["output"]
total_cost  = input_cost + output_cost

# ── Graph stats ───────────────────────────────────────────────────────
total_nodes = sum(len(gd.nodes)         for gd in graph_documents)
total_edges = sum(len(gd.relationships) for gd in graph_documents)
node_types  = set(n.type for gd in graph_documents for n in gd.nodes)
edge_types  = set(r.type for gd in graph_documents for r in gd.relationships)

# ── Report ────────────────────────────────────────────────────────────
print(f"\n  ⏱  Build time          : {elapsed:.2f}s  ({elapsed/60:.1f} min)")
print(f"  📄 Documents processed : {len(documents)}")
print(f"  🔵 Nodes extracted     : {total_nodes}  ({len(node_types)} types: {', '.join(sorted(node_types))})")
print(f"  🔗 Relationships       : {total_edges}  ({len(edge_types)} types)")
print()
print(f"  📥 Est. input tokens   : {est_input_tokens:,}")
print(f"  📤 Est. output tokens  : {est_output_tokens:,}")
print(f"  🔢 Total tokens        : {est_input_tokens + est_output_tokens:,}")
print()
print(f"  💵 Input cost          : ${input_cost:.6f}")
print(f"  💵 Output cost         : ${output_cost:.6f}")
print(f"  💰 TOTAL COST          : ${total_cost:.6f}  (model: {MODEL})")
print()
print(f"  📊 Cost per document   : ${total_cost/len(documents):.6f}")
print(f"  📊 Cost per 1K tokens  : ${(total_cost/(est_input_tokens+est_output_tokens))*1000:.6f}")
print("=" * 60)

# ── Store for downstream use ──────────────────────────────────────────
build_stats = {
    "model":              MODEL,
    "elapsed_s":          round(elapsed, 3),
    "documents":          len(documents),
    "nodes":              total_nodes,
    "relationships":      total_edges,
    "est_input_tokens":   est_input_tokens,
    "est_output_tokens":  est_output_tokens,
    "total_tokens":       est_input_tokens + est_output_tokens,
    "input_cost_usd":     round(input_cost,  8),
    "output_cost_usd":    round(output_cost, 8),
    "total_cost_usd":     round(total_cost,  8),
}
print(f"\n  build_stats dict available for downstream cells.")

  GraphRAG — Knowledge Graph Build (Cost Tracker)

  ⏱  Build time          : 56.34s  (0.9 min)
  📄 Documents processed : 14
  🔵 Nodes extracted     : 56  (18 types: Amount, City, Company, Concept, Country, Date, Enterprise, Humanoid robot, Institution, Location, Nationality, Organization, Person, Product, Program, Technology, University, Year)
  🔗 Relationships       : 43  (23 types)

  📥 Est. input tokens   : 491
  📤 Est. output tokens  : 2,675
  🔢 Total tokens        : 3,166

  💵 Input cost          : $0.000074
  💵 Output cost         : $0.001605
  💰 TOTAL COST          : $0.001679  (model: gpt-4o-mini)

  📊 Cost per document   : $0.000120
  📊 Cost per 1K tokens  : $0.000530

  build_stats dict available for downstream cells.
